# 맨홀-노드 GNN 그래프 v3 — 정본 통합

08(원본)·09(방향보정)·내 정제(좌표교정·공간정합)를 **하나의 정본**으로 통합.
- **노드 좌표 = 공식 맨홀(sb101) 우선**(폴백 관거끝점) — 좌표 불일치·장거리 엣지 해소
- **엣지 방향 = 관저고 보정**(시작→끝과 반대면 반전, `direction_confidence`)
- **엣지 품질 = 공간정합 플래그**(노드거리 ≤ 관거연장×1.5+20) — 제거 아닌 플래그
- **학습 산출물 = node_idx · edge_index.npy · sensor_snap** + raw/집계 엣지 분리
- **검증 = 공식 접속관수(CN_PE_NUM) vs 그래프 차수**

In [1]:
import os; os.chdir('/home/namjun/city_flood')
import sys; sys.path.insert(0,'scripts'); from krfont import set_korean; set_korean()
import geopandas as gpd, pandas as pd, numpy as np, networkx as nx, warnings
from scipy.spatial import cKDTree
warnings.filterwarnings('ignore')
base="03_GIS/관악구_하수관로_맨홀_shp/"; EB="dataset/processed/eda_based/"
def dec(x):
    try: return x.encode('latin1').decode('cp949')
    except: return x
pipe=gpd.read_file(base+"sb001.shp")
p=pipe[pipe.sat_mhe.notna()&pipe.end_mhe.notna()&(pipe.sat_mhe.astype(str).str.strip()!='')&(pipe.end_mhe.astype(str).str.strip()!='')].copy()
p['pipe_id']=p.index.astype(str); p['sat']=p.sat_mhe.astype(str).str.strip(); p['end']=p.end_mhe.astype(str).str.strip()
p['소유역']=p.swa_nam.map(dec)
p['length']=pd.to_numeric(p.lenx,errors='coerce').where(lambda s:s>0)
p['si']=pd.to_numeric(p.st_pip_hit,errors='coerce'); p['ei']=pd.to_numeric(p.et_pip_hit,errors='coerce')
p['raw_slope']=np.where((p.length>0)&(p.si>0)&(p.ei>0),(p.si-p.ei)/p.length,np.nan)
p['has_inv']=(p.length>0)&(p.si>0)&(p.ei>0)&p.raw_slope.notna()
print("엣지화 가능 관거",len(p))

엣지화 가능 관거 16059


## 1. 노드 좌표 — 공식 맨홀(sb101) 우선, 폴백 관거끝점

In [2]:
mh=gpd.read_file(base+"sb101.shp").drop_duplicates('mhe_idn'); mh['mhe_idn']=mh.mhe_idn.astype(str)
coord={r.mhe_idn:(r.geometry.x,r.geometry.y) for _,r in mh.iterrows()}
ep={}
for _,r in p.iterrows():
    c=list(r.geometry.coords); ep.setdefault(r.sat,c[0]); ep.setdefault(r.end,c[-1])
allids=set(p.sat)|set(p.end); xy={k:(coord.get(k) or ep[k]) for k in allids}
n_off=sum(1 for k in allids if k in coord)
print(f"노드 {len(allids)} | 공식맨홀 좌표 {n_off}({n_off/len(allids):.0%}) · 폴백 {len(allids)-n_off}")

노드 13272 | 공식맨홀 좌표 9652(73%) · 폴백 3620


## 2. 엣지 — 관저고 방향보정 + 공간정합 플래그

In [3]:
p['is_flat']=p.has_inv & np.isclose(p.raw_slope,0)
p['is_reversed']=p.has_inv & (p.raw_slope<0)
p['src']=np.where(p.is_reversed,p.end,p.sat); p['dst']=np.where(p.is_reversed,p.sat,p.end)
p['flow_slope']=np.where(p.has_inv,np.abs(p.raw_slope),np.nan)
p['direction_confidence']=np.select([p.is_reversed,p.has_inv&(p.raw_slope>0),p.is_flat],
    ['reversed_by_invert','official_matches_invert','flat_keep_official'],default='official_only_missing_invert')
sx=np.array([xy[s] for s in p.src]); dx=np.array([xy[d] for d in p.dst])
p['nd_dist']=np.hypot(sx[:,0]-dx[:,0],sx[:,1]-dx[:,1])
L=p.length.fillna(p.nd_dist); p['공간정합']=(p.nd_dist<=L*1.5+20)
edges_raw=p[['pipe_id','sat','end','src','dst','length','raw_slope','flow_slope','is_reversed','direction_confidence','nd_dist','공간정합','소유역']].copy()
edge_nx=edges_raw.groupby(['src','dst'],as_index=False).agg(
    n_pipes=('pipe_id','count'),length=('length','sum'),mean_flow_slope=('flow_slope','mean'),
    reversed_count=('is_reversed','sum'),nd_dist=('nd_dist','first'),공간정합=('공간정합','all'),
    confidence_modes=('direction_confidence',lambda s:'|'.join(sorted(set(map(str,s))))),
    소유역=('소유역',lambda s:next((x for x in s if pd.notna(x)),np.nan)))
print(f"관거레코드 {len(edges_raw)} → 집계엣지 {len(edge_nx)} | 방향반전 {int(edges_raw.is_reversed.sum())} | 공간정합(집계) {edge_nx.공간정합.mean():.0%}")
print("방향판정:",edges_raw.direction_confidence.value_counts().to_dict())

관거레코드 16059 → 집계엣지 15260 | 방향반전 1047 | 공간정합(집계) 78%
방향판정: {'official_matches_invert': 14688, 'reversed_by_invert': 1047, 'flat_keep_official': 214, 'official_only_missing_invert': 110}


## 3. 노드 속성·node_idx + 공식 접속관수 검증

In [4]:
nodes=pd.DataFrame([(k,xy[k][0],xy[k][1]) for k in allids],columns=['node_id','x','y'])
nodes=nodes.merge(mh[['mhe_idn','hsl','cn_pe_num','wr_rn_cde','me_in_cde']].rename(columns={'mhe_idn':'node_id'}),on='node_id',how='left')
nodes['소유역']=nodes.node_id.map(mh.set_index('mhe_idn').swa_nam.map(dec))
G=nx.DiGraph(); [G.add_edge(a,b) for a,b in edge_nx[['src','dst']].values]
nodes['degree']=nodes.node_id.map({n:G.in_degree(n)+G.out_degree(n) for n in G.nodes}).fillna(0).astype(int)
nodes=nodes.sort_values('node_id').reset_index(drop=True); nodes['node_idx']=np.arange(len(nodes))
v=nodes[(nodes.cn_pe_num.notna())&(nodes.cn_pe_num>0)]
print(f"검증(접속관수): 정확 {(v.cn_pe_num==v.degree).mean():.0%}·±1 {(abs(v.cn_pe_num-v.degree)<=1).mean():.0%} | 약연결성분 {nx.number_weakly_connected_components(G)}")

검증(접속관수): 정확 80%·±1 90% | 약연결성분 96


## 4. 센서 snap + edge_index + 저장

In [5]:
nidx=nodes.set_index('node_id').node_idx.to_dict()
gs=gpd.read_file("03_GIS/하수관로_노면_수위계_shp/도시침수계(관로)_서울시 2026.shp")
gs['자치구']=gs['자치구'].astype(str); gs=gs[gs.자치구.str.contains('관악')].to_crs(5181)
have=set(pd.read_parquet("dataset/processed/cleaned/sewer_node.parquet",columns=['sensor_id']).sensor_id)
tree=cKDTree(nodes[['x','y']].values); d,idx=tree.query(np.array([(g.x,g.y) for g in gs.geometry]))
snap=pd.DataFrame({'sensor_id':gs['수위계번호'].astype(str).values,'node_id':nodes.node_id.values[idx],'snap_m':d})
snap['node_idx']=snap.node_id.map(nidx); snap['is_observed']=snap.sensor_id.isin(have)
nodes=nodes.merge(snap[['node_id','sensor_id','is_observed','snap_m']],on='node_id',how='left'); nodes['is_sensor']=nodes.sensor_id.notna()
edge_nx['src_idx']=edge_nx.src.map(nidx); edge_nx['dst_idx']=edge_nx.dst.map(nidx)
edge_index=edge_nx[['src_idx','dst_idx']].to_numpy().T.astype(np.int64)
nodes.to_parquet(EB+"gnn_manhole_nodes.parquet",index=False)
edges_raw.to_parquet(EB+"gnn_manhole_edges_raw.parquet",index=False)
edge_nx.to_parquet(EB+"gnn_manhole_edges.parquet",index=False)
snap.to_parquet(EB+"gnn_manhole_sensor_snap.parquet",index=False)
np.save(EB+"gnn_manhole_edge_index.npy",edge_index)
print(f"센서 snap 42 (중앙 {np.median(d):.0f}m) 보유 {int(snap.is_observed.sum())}/신설 {int((~snap.is_observed).sum())}")
print("저장 완료: nodes/edges_raw/edges/sensor_snap/edge_index.npy (정본 v3)")

센서 snap 42 (중앙 9m) 보유 13/신설 29
저장 완료: nodes/edges_raw/edges/sensor_snap/edge_index.npy (정본 v3)


## 결론 (정본 v3)
- 좌표(공식맨홀 73%)·방향(관저고 보정 1,047반전)·품질(공간정합 78%)·검증(접속관수 80%/±1 90%)·학습인덱스를 **한 산출물로 통합**.
- 08/09가 덮어쓰던 충돌 해소. 이후 STGNN(10/최적화판)은 이 정본을 그대로 사용.
- 다음: STGNN 깨끗한 재실행(relay broadcast 최적화·풀 시드) → GNN-off 이상치 규명.